# TransferIa en Google Colab

Cambio de cara con formularios, como la interfaz del programa. Corre las celdas **en orden**; cada una tiene su formulario a la derecha (o arriba en el celular).

1. **Instalar** (una vez por sesión) · 2. **Modelos** · 3. **Subir archivos** (opcional)
4. **Crear modelo de cara** (opcional) · 5. **Ver personas** (opcional, modo identidad)
6. **Procesar** · 7. **Ver y descargar el resultado**

Antes de empezar: **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU (T4)**. Sin GPU funciona igual, pero mucho más lento.

Usa TransferIa solo con fotos y videos de personas que dieron su permiso.

In [ ]:
#@title 1. Instalar TransferIa { display-mode: "form" }
#@markdown Descarga el programa de GitHub e instala lo necesario. Una vez por sesión (~2 min).
repositorio = "https://github.com/JuniorHZ19/TransferIA2.git" #@param {type:"string"}
rama = "main" #@param {type:"string"}

import os, subprocess, sys

RAIZ = "/content/TransferIA2"


def sh(comando, mostrar=True):
    r = subprocess.run(comando, shell=True, text=True, capture_output=True)
    if r.returncode != 0:
        print(r.stdout[-2000:], r.stderr[-2000:])
        raise RuntimeError(f"Falló: {comando}")
    if mostrar and r.stdout.strip():
        print(r.stdout.strip()[-1000:])
    return r.stdout


if os.path.isdir(f"{RAIZ}/.git"):
    sh(f"git -C {RAIZ} fetch -q origin {rama} && git -C {RAIZ} checkout -q {rama} && git -C {RAIZ} pull -q origin {rama}")
    print("Programa actualizado")
else:
    sh(f"git clone -q -b {rama} {repositorio} {RAIZ}")
    print("Programa descargado")
os.chdir(RAIZ)
print("Versión:", sh(f"git -C {RAIZ} log -1 --format='%h %s'", mostrar=False).strip())

print("Instalando librerías...")
sh("pip uninstall -y -q onnxruntime", mostrar=False)
sh("pip install -q insightface onnxruntime-gpu pyzipper", mostrar=False)

import onnxruntime as ort
if "CUDAExecutionProvider" in ort.get_available_providers():
    print("Listo. GPU activa (CUDA)")
else:
    print("Listo, pero SIN GPU: Entorno de ejecución → Cambiar tipo → GPU T4 (será mucho más lento)")

In [ ]:
#@title 2. Modelos { display-mode: "form" }
#@markdown Los modelos no están en GitHub (pesan ~700 MB). Elige de dónde traerlos:
origen = "Drive: zip" #@param ["Drive: zip", "Drive: carpeta", "Subir zip"]
#@markdown Ruta en tu Drive del zip (`modelsTransferIa.zip`) o de la carpeta con los `.onnx`:
ruta_drive = "/content/drive/MyDrive/modelsTransferIa.zip" #@param {type:"string"}

import os, shutil, zipfile

os.chdir(RAIZ)
os.makedirs("models", exist_ok=True)


def montar_drive():
    if not os.path.isdir("/content/drive/MyDrive"):
        from google.colab import drive
        drive.mount("/content/drive")


def descomprimir(ruta_zip):
    with zipfile.ZipFile(ruta_zip) as z:
        for info in z.infolist():
            if info.filename.lower().endswith((".onnx", ".pth")):
                with z.open(info) as entrada, open(os.path.join("models", os.path.basename(info.filename)), "wb") as salida:
                    shutil.copyfileobj(entrada, salida)


if origen == "Subir zip":
    from google.colab import files
    for nombre in files.upload():
        descomprimir(nombre)
        os.remove(nombre)
else:
    montar_drive()
    if not os.path.exists(ruta_drive):
        raise FileNotFoundError(f"No existe en tu Drive: {ruta_drive}")
    if origen == "Drive: zip":
        print("Descomprimiendo modelos...")
        descomprimir(ruta_drive)
    else:
        for nombre in os.listdir(ruta_drive):
            if nombre.lower().endswith((".onnx", ".pth")):
                shutil.copy(os.path.join(ruta_drive, nombre), "models")

necesarios = {"inswapper128.onnx": "cambio de cara", "face_occluder.onnx": "oclusión"}
opcionales = {"gpen_bfr_256.onnx": "mejora GPEN"}
faltan = [n for n in necesarios if not os.path.exists(f"models/{n}")]
for nombre, uso in {**necesarios, **opcionales}.items():
    ok = os.path.exists(f"models/{nombre}")
    print(f"{'OK   ' if ok else 'FALTA'} {nombre} ({uso})")
if faltan:
    raise FileNotFoundError(f"Faltan modelos necesarios: {', '.join(faltan)}")
print("Modelos listos")

In [ ]:
#@title 3. Subir archivos (opcional) { display-mode: "form" }
#@markdown Sube fotos, videos o modelos `.npy` desde tu PC. También puedes usar archivos de tu Drive
#@markdown directamente (por ejemplo `/content/drive/MyDrive/videos/clienta.mp4`) sin subirlos aquí.
carpeta = "/content/entradas" #@param {type:"string"}

import os
from google.colab import files

os.makedirs(carpeta, exist_ok=True)
subidos = files.upload()
for nombre, datos in subidos.items():
    ruta = os.path.join(carpeta, nombre)
    with open(ruta, "wb") as f:
        f.write(datos)
    if os.path.abspath(nombre) != os.path.abspath(ruta) and os.path.exists(nombre):
        os.remove(nombre)
    print("Copia esta ruta en el formulario:", ruta)

In [ ]:
#@title 4. Crear modelo de cara (opcional) { display-mode: "form" }
#@markdown Junta varias fotos y/o videos de la MISMA persona en un modelo `.npy` (más parecido y estable que una sola foto).
#@markdown Úsalo después como **fuente** en el paso 6.
nombre = "persona1" #@param {type:"string"}
#@markdown Fotos o carpetas con fotos, separadas por comas (vacío = ninguna):
fotos = "" #@param {type:"string"}
#@markdown Videos de la persona, separados por comas (vacío = ninguno):
videos = "" #@param {type:"string"}
#@markdown Frames por segundo que se sacan de cada video (0.5 = uno cada 2 segundos):
fps_video = 1 #@param {type:"number"}
#@markdown Dónde guardarlo (en Drive queda para la próxima sesión):
carpeta_modelos = "/content/drive/MyDrive/TransferIa/modelos_caras" #@param {type:"string"}

import os, subprocess, sys
from IPython.display import display, Image

os.chdir(RAIZ)
if carpeta_modelos.startswith("/content/drive") and not os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")
lista = lambda texto: [p.strip() for p in texto.split(",") if p.strip()]
if not lista(fotos) and not lista(videos):
    raise ValueError("Pon al menos una foto, carpeta o video")
os.makedirs(carpeta_modelos, exist_ok=True)
salida = os.path.join(carpeta_modelos, f"{nombre}.npy")
comando = [sys.executable, "cli.py", "crear-modelo", "--output", salida, "--overwrite", "--fps", str(fps_video)]
if lista(fotos):
    comando += ["--fotos", *lista(fotos)]
if lista(videos):
    comando += ["--video", *lista(videos)]
r = subprocess.run(comando, capture_output=True, encoding="utf-8", errors="replace",
                   env={**os.environ, "PYTHONIOENCODING": "utf-8"})
print("\n".join(l for l in (r.stdout + r.stderr).splitlines() if "Warning" not in l and "warn" not in l)[-3000:])
if r.returncode == 0 and os.path.exists(salida[:-4] + ".jpg"):
    display(Image(salida[:-4] + ".jpg", width=160))
    print("Fuente para el paso 6:", salida)

In [ ]:
#@title 5. Ver personas (opcional, para modo identidad) { display-mode: "form" }
#@markdown Analiza el video destino y muestra quién es cada número: **0 = la persona que más aparece**.
#@markdown Ese número va en **indices_destino** del paso 6. El análisis queda guardado y el paso 6 lo reutiliza
#@markdown (con la misma carpeta de salida, género y opciones).
destino = "/content/entradas/video.mp4" #@param {type:"string"}
genero = "todos" #@param ["todos", "mujer", "hombre"]
personas_fps = 1 #@param {type:"number"}
personas_max = 300 #@param {type:"integer"}
#@markdown Carpeta de salida (la misma que usarás en el paso 6):
carpeta_salida = "output" #@param {type:"string"}

import os, sys
import cv2
import matplotlib.pyplot as plt

os.chdir(RAIZ)
sys.path.insert(0, RAIZ)
from core.Face_Swaper import Swapper_Faces
from core.Personas import Personas, validar_fps, validar_maximo

if "_swapper_personas" not in globals():
    _swapper_personas = Swapper_Faces(os.path.join("models", "inswapper128.onnx"))


def avance(hechos, total):
    print(f"\rAnalizando: frame {hechos}/~{total}", end="")


personas, carpeta, reutilizado = Personas.obtener(
    destino, carpeta_salida, lambda frame: _swapper_personas.detectar_caras(frame, genero), genero,
    progreso=avance, fps=validar_fps(personas_fps), maximo=validar_maximo(personas_max))
print("\n" + ("Análisis guardado (no se repitió)" if reutilizado else "Análisis listo"))
if not len(personas):
    print("No se encontró ninguna cara en el video")
else:
    mostrar = personas.lista[:12]
    columnas = min(4, len(mostrar))
    filas = (len(mostrar) + columnas - 1) // columnas
    fig, ejes = plt.subplots(filas, columnas, figsize=(3 * columnas, 3.4 * filas), squeeze=False)
    for eje in ejes.flat:
        eje.axis("off")
    for i, p in enumerate(mostrar):
        eje = ejes.flat[i]
        if p["miniatura"] is not None:
            eje.imshow(cv2.cvtColor(p["miniatura"], cv2.COLOR_BGR2RGB))
        desde = f"{int(p['desde'] // 60)}:{int(p['desde'] % 60):02d}"
        eje.set_title(f"Persona {i}\naparece {p['apariciones']} veces · desde {desde}", fontsize=10)
    plt.tight_layout()
    plt.show()
    if len(personas) > len(mostrar):
        print(f"... y {len(personas) - len(mostrar)} más con pocas apariciones (extras)")

In [ ]:
#@title 6. Procesar { display-mode: "form" }
#@markdown ### Archivos
#@markdown Fuente: foto, o modelo(s) `.npy` separados por comas. Destino: imagen, video o carpeta de imágenes.
fuente = "/content/entradas/cara.jpg" #@param {type:"string"}
destino = "/content/entradas/video.mp4" #@param {type:"string"}
#@markdown Salida sin extensión (con una carpeta de imágenes, es la carpeta donde se crea `<carpeta>_output`):
salida = "output/resultado" #@param {type:"string"}
formato_video = "mp4" #@param ["mp4", "mkv", "mov"]
#@markdown ---
#@markdown ### Caras
asignacion = "posicion" #@param ["posicion", "identidad"]
genero = "todos" #@param ["todos", "mujer", "hombre"]
#@markdown Índices: `0,0` y `0,1` = la misma fuente en dos caras. En identidad con video, destino = número de persona (paso 5).
indices_fuente = "0" #@param {type:"string"}
indices_destino = "0" #@param {type:"string"}
#@markdown Análisis de personas (solo identidad en video):
personas_fps = 1 #@param {type:"number"}
personas_max = 300 #@param {type:"integer"}
#@markdown ---
#@markdown ### Pegado
mascara = "amplia" #@param ["amplia", "normal", "maxima"]
correccion_color = "ninguna" #@param ["ninguna", "suave", "completa"]
#@markdown ---
#@markdown ### Del destino (lo que se queda como en el video original)
ojos = False #@param {type:"boolean"}
boca = False #@param {type:"boolean"}
nariz = False #@param {type:"boolean"}
frente_quitar_cerquillo = False #@param {type:"boolean"}
maquillaje = 0 #@param {type:"slider", min:0, max:1, step:0.05}
#@markdown ---
#@markdown ### Oclusión y nitidez
oclusion = True #@param {type:"boolean"}
oclusion_modo = "adaptable" #@param ["adaptable", "debil", "fuerte"]
nitidez = "None" #@param ["None", "natural", "strong", "adaptativo"]
#@markdown ---
#@markdown ### Mejora (GPEN)
mejorar_gpen = False #@param {type:"boolean"}
fuerza_gpen = 0.8 #@param {type:"slider", min:0.1, max:1, step:0.05}
#@markdown Cara mínima: píxeles (`96`) o porcentaje de la imagen (`25%`):
cara_minima = "96" #@param {type:"string"}
#@markdown ---
#@markdown ### Video
hilos = 4 #@param {type:"integer"}
codec = "h264_nvenc" #@param ["h264_nvenc", "libx264"]
#@markdown Calidad (menor = mejor y más pesado):
calidad_crf = 18 #@param {type:"slider", min:12, max:30, step:1}
video_temporal = "mkv" #@param ["mkv", "mp4frag"]
#@markdown Fragmento (vacío = video completo); segundos o `m:ss`:
inicio = "" #@param {type:"string"}
fin = "" #@param {type:"string"}
sin_audio = False #@param {type:"boolean"}
#@markdown Si quedó un proceso a medias de esta misma salida:
si_quedo_a_medias = "continuar" #@param ["continuar", "empezar de cero"]
#@markdown ---
#@markdown ### Salida
sobrescribir = True #@param {type:"boolean"}
guardar_en_zip_con_contrasena = False #@param {type:"boolean"}
conservar_archivo_suelto = False #@param {type:"boolean"}
#@markdown ---
#@markdown ### Configuraciones guardadas (opcional)
#@markdown Usar una guardada (las opciones de este formulario tienen prioridad) / guardar las de este formulario:
usar_configuracion = "" #@param {type:"string"}
guardar_configuracion = "" #@param {type:"string"}

import os, subprocess, sys, getpass

os.chdir(RAIZ)
EXT_IMAGEN = (".jpg", ".jpeg", ".png", ".bmp", ".webp")
if destino.startswith("/content/drive") and not os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")
if not os.path.exists(destino):
    raise FileNotFoundError(f"No existe el destino: {destino}")

# h264_nvenc necesita la GPU NVIDIA: se prueba antes, y si no anda se usa libx264 (CPU)
if codec == "h264_nvenc":
    prueba = subprocess.run(["ffmpeg", "-v", "error", "-f", "lavfi", "-i", "color=size=256x256:duration=0.2",
                             "-c:v", "h264_nvenc", "-f", "null", "-"], capture_output=True, text=True)
    if prueba.returncode != 0:
        print("Aviso: h264_nvenc no funciona en esta sesión (¿sin GPU? Entorno de ejecución → Cambiar tipo → T4).\n"
              "Se usa libx264 (CPU): funciona igual, pero más lento.\n")
        codec = "libx264"

if os.path.isdir(destino):
    ruta_salida = salida  # carpeta donde se crea <carpeta>_output
elif destino.lower().endswith(EXT_IMAGEN):
    ruta_salida = f"{salida}{os.path.splitext(destino)[1].lower()}"
else:
    ruta_salida = f"{salida}.{formato_video}"
os.makedirs(os.path.dirname(os.path.abspath(ruta_salida)) if not os.path.isdir(destino) else ruta_salida, exist_ok=True)

c = [sys.executable, "-u", "cli.py", "process", "--source", fuente, "--target", destino, "--output", ruta_salida,
     "--asignacion", asignacion, "--genero", genero, "--source-index", indices_fuente.replace(" ", ""),
     "--target-index", indices_destino.replace(" ", ""), "--personas-fps", str(personas_fps),
     "--personas-max", str(personas_max), "--face-mask", mascara, "--color", correccion_color,
     "--nitidez", nitidez, "--oclusion-modo", oclusion_modo, "--threads", str(hilos), "--video-codec", codec,
     "--crf", str(calidad_crf), "--container", video_temporal]
c += ["--eyes"] * ojos + ["--mouth"] * boca + ["--nose"] * nariz + ["--remove-bangs"] * frente_quitar_cerquillo
c += ["--occlusion"] * oclusion + ["--no-audio"] * sin_audio + ["--overwrite"] * sobrescribir
if maquillaje > 0:
    c += ["--makeup", str(maquillaje)]
if mejorar_gpen:
    c += ["--enhance", str(fuerza_gpen), "--enhance-min-size", cara_minima.strip()]
if inicio.strip():
    c += ["--start", inicio.strip()]
if fin.strip():
    c += ["--end", fin.strip()]
a_medias = os.path.isdir(os.path.abspath(ruta_salida) + ".partes")  # lo hecho de un video que se corto
c += ["--resume"] if a_medias and si_quedo_a_medias == "continuar" else ["--restart"]
if usar_configuracion.strip():
    c += ["--config", usar_configuracion.strip()]
if guardar_configuracion.strip():
    c += ["--save-config", guardar_configuracion.strip()]
if guardar_en_zip_con_contrasena:
    contrasena = getpass.getpass("Contraseña del ZIP (no se muestra ni se guarda): ")
    if not contrasena or getpass.getpass("Repítela: ") != contrasena:
        raise ValueError("Contraseña vacía o distinta; vuelve a correr la celda")
    c += ["--zip", "--zip-password", contrasena] + ["--zip-conservar"] * conservar_archivo_suelto


def correr(comando):
    proceso = subprocess.Popen(comando, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1,
                               encoding="utf-8", errors="replace", env={**os.environ, "PYTHONIOENCODING": "utf-8"})
    salida_cli = []
    for linea in iter(proceso.stdout.readline, ""):
        if "Warning" in linea or "warnings.warn" in linea or "Applied providers" in linea or "find model" in linea:
            continue
        salida_cli.append(linea)
        print(linea, end="")
    return proceso.wait(), "".join(salida_cli)


visible = ["***" if i > 0 and c[i - 1] == "--zip-password" else p for i, p in enumerate(c)]
print("Comando:", " ".join(visible[2:]), "\n")
if "--resume" in c:
    print("Hay un proceso a medias de esta salida: se continúa desde donde quedó\n")
codigo, texto = correr(c)
if codigo != 0 and "--resume" in c and ("No se puede continuar" in texto or "No hay un proceso sin terminar" in texto):
    # Lo hecho se hizo con otras opciones (por ejemplo otro códec), o no quedó nada que continuar
    print("\nNo se puede continuar lo que quedó a medias (otras opciones o incompleto): se empieza de cero\n")
    codigo, texto = correr(["--restart" if p == "--resume" else p for p in c])

ULTIMO_RESULTADO = None
for linea in texto.splitlines():
    if "Resultado guardado en:" in linea:
        ULTIMO_RESULTADO = linea.split("Resultado guardado en:", 1)[1].split(" (con contrase")[0].strip()
print("\nListo:" if codigo == 0 else "\nTerminó con error (arriba está el motivo)", ULTIMO_RESULTADO or "")

In [ ]:
#@title 7. Ver y descargar el resultado { display-mode: "form" }
#@markdown Vacío = el último resultado del paso 6.
archivo = "" #@param {type:"string"}
mostrar_aqui = True #@param {type:"boolean"}
descargar_a_mi_pc = False #@param {type:"boolean"}
#@markdown Copiar también a tu Drive (vacío = no copiar):
copiar_a_drive = "" #@param {type:"string"}

import os, shutil

os.chdir(RAIZ)
ruta = archivo.strip() or (ULTIMO_RESULTADO if "ULTIMO_RESULTADO" in globals() else None)
if not ruta or not os.path.exists(ruta):
    raise FileNotFoundError(f"No encuentro el resultado: {ruta}")
tamano = (sum(os.path.getsize(os.path.join(d, f)) for d, _, fs in os.walk(ruta) for f in fs)
          if os.path.isdir(ruta) else os.path.getsize(ruta))
print(f"{ruta} ({tamano / 1e6:.1f} MB)")

if mostrar_aqui and os.path.isfile(ruta):
    from IPython.display import display, Image, Video
    if ruta.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".webp")):
        display(Image(ruta, width=640))
    elif ruta.lower().endswith((".mp4", ".mov")):
        if tamano < 60e6:
            display(Video(ruta, embed=True, width=640))
        else:
            print("Video grande para verlo aquí: descárgalo o cópialo a Drive")
    elif ruta.lower().endswith(".zip"):
        print("Está en ZIP con contraseña: se abre con 7-Zip o WinRAR")

if os.path.isdir(ruta):  # carpeta de imagenes: se descarga como zip (sin contraseña)
    ruta = shutil.make_archive(ruta, "zip", ruta)
if copiar_a_drive.strip():
    if not os.path.isdir("/content/drive/MyDrive"):
        from google.colab import drive
        drive.mount("/content/drive")
    os.makedirs(copiar_a_drive, exist_ok=True)
    print("Copiado a:", shutil.copy(ruta, copiar_a_drive))
if descargar_a_mi_pc:
    from google.colab import files
    files.download(ruta)